# Wire CodePipeline to trigger a SageMaker Pipelines retraining run on commit, with EventBridge surfacing approval events and CloudWatch catching failures

A platform team's data scientists are committing model code to CodeCommit and manually kicking off SageMaker training runs by hand. No audit trail, no rollback, no automation. The CISO and the head of ML want CI/CD on every retraining run.

You have one session to wire CodePipeline plus CodeBuild plus CodeCommit plus SageMaker Pipelines plus EventBridge plus CloudWatch into one chain. Commit code, watch CodePipeline trigger, watch CodeBuild upsert the SageMaker Pipeline, watch the SageMaker Pipeline train and register, watch EventBridge emit on the model-registry event, watch the approval action gate the (hypothetical) deployment, and watch the CloudWatch alarm sit ready to fire if anything fails.

Five artifacts ship in this lab:

- A CodeCommit repository with `buildspec.yml` and `pipeline_definition.py` seeded on the main branch.
- A CodeBuild project that runs the buildspec to upsert and start the SageMaker Pipeline.
- A CodePipeline with three stages (Source, Build, Approve).
- An EventBridge rule that matches SageMaker Model Package state changes and writes to a CloudWatch log group.
- A CloudWatch alarm on the CodePipeline failure metric.

**Time.** Plan for about 75 minutes hands-on. CodeCommit + CodeBuild + CodePipeline creation is about 5 minutes. The first CodePipeline run end-to-end (CodeBuild + 4-step SageMaker Pipeline + Approve gate) takes about 12 to 15 minutes. The manual approval simulation is instant. Cleanup is multi-step and takes 5 to 10 minutes because of the service breadth. Budget 150 minutes if anything fights back.

**Cost.** This lab costs about 5 to 10 cents per session. CodePipeline is $1 per active pipeline per month flat. Prorated to a 90-minute session that is $0.002. CodeBuild on general1.small at $0.005 per build-minute for about 3 minutes is $0.015. CodeCommit is free for the first 5 active users; the lab uses 1. The SageMaker Pipelines step inside the pipeline adds another roughly $0.06 per triggered run (4 steps on ml.m5.large at $0.115/hour for ~3 minutes each). EventBridge is free at this volume. CloudWatch alarm is free (10 free per month). SNS topic is free. The residual that matters: CodePipeline at $1 per month flat keeps billing until `delete_pipeline` is called; the cleanup cell calls that explicitly.

This lab maps to AWS MLA-C01 Domain 3 (Deployment and Orchestration of ML Workflows, 22%) on CI/CD for ML (CodePipeline + CodeBuild + CodeCommit + SageMaker Pipelines) AND Domain 4 (ML Solution Monitoring, Maintenance, and Security, 24%) on EventBridge-driven retraining and CloudWatch monitoring. It is the cert-track capstone, the highest service count of any MLA lab.

**CodeCommit note.** CodeCommit is closed to new customers. Existing customers with CodeCommit already enabled can run this lab as written; new customers should treat the CodeCommit step as a stand-in for any source control that CodeStar Connections supports (GitHub, GitLab, Bitbucket). The exam pattern is the same.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern.

import atexit
import getpass
import json
import sys
import time
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "mlops-cicd-end-to-end"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

REPO_NAME = f"labrun-{LAB_ID}-repo"
BUILD_PROJECT = f"labrun-{LAB_ID}-build"
PIPELINE_NAME = f"labrun-{LAB_ID}-pipeline"
PIPELINE_ROLE_NAME = f"labrun-{LAB_ID}-pipeline-role"
BUILD_ROLE_NAME = f"labrun-{LAB_ID}-build-role"
SM_ROLE_NAME = f"labrun-{LAB_ID}-sm-role"
PIPELINE_INLINE_POLICY = "labrun-mla-lab12-pipeline-policy"
BUILD_INLINE_POLICY = "labrun-mla-lab12-build-policy"
SM_INLINE_POLICY = "labrun-mla-lab12-sm-policy"
SM_PIPELINE_NAME = f"labrun-{LAB_ID}-sm-pipeline"
PACKAGE_GROUP_NAME = f"labrun-{LAB_ID}-group"
EB_RULE_NAME = f"labrun-{LAB_ID}-rule"
LOG_GROUP_NAME = f"/aws/labrun/mla/lab12/events"
ALARM_NAME = f"labrun-{LAB_ID}-failure-alarm"
SNS_TOPIC_NAME = f"labrun-{LAB_ID}-alarm-topic"

BUCKET_NAME = None
ACCOUNT_ID = None
PIPELINE_ROLE_ARN = None
BUILD_ROLE_ARN = None
SM_ROLE_ARN = None
SNS_TOPIC_ARN = None
REPO_CLONE_URL_HTTP = None
PIPELINE_EXECUTION_ID = None
SESSION_START = datetime.now(timezone.utc)

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 3 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
PIPELINE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{PIPELINE_ROLE_NAME}"
BUILD_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{BUILD_ROLE_NAME}"
SM_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SM_ROLE_NAME}"
SNS_TOPIC_ARN = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{SNS_TOPIC_NAME}"
print(f"Bucket: {BUCKET_NAME}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + orphan scan. labrun-checks 0.6.0 has no
# adapters for CodePipeline, CodeBuild project, CodeCommit repository,
# SageMaker Pipeline, Model Package, Model Package Group, EventBridge
# rule, CloudWatch alarm, CloudWatch log group, or SNS topic. The cleanup
# cell tears those down manually BEFORE run_cleanup walks the manifest.
# The manifest below contains only adapter-supported types.
#
# Critical resources: in-flight CodePipeline executions (CodeBuild + SM
# Pipeline steps continue to bill until stopped), and the CodePipeline
# itself ($1/pipeline/month flat).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=PIPELINE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {PIPELINE_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=BUILD_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {BUILD_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=SM_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {SM_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the CodeCommit repository and seed it with `buildspec.yml` and `pipeline_definition.py` on the main branch

The repo is the source of truth for the retraining workflow. Two files seed the main branch:

- `buildspec.yml` instructs CodeBuild to install the SageMaker Python SDK and run `python pipeline_definition.py upsert-and-start`.
- `pipeline_definition.py` defines a four-step SageMaker Pipeline (Process, Train, Evaluate, Register) and exposes a CLI entry point.

The lab uses CodeCommit's PutFile API rather than git push because the Colab kernel does not have git pre-wired against CodeCommit credentials.

Build it in this order:

1. Create the supporting IAM roles (pipeline, build, sagemaker) and S3 bucket.
2. Create the CodeCommit repository.
3. Use `put_file` to seed `buildspec.yml` and `pipeline_definition.py` to main.

Your job: write the two `put_file` calls.

In [ ]:
# NBVAL_SKIP
# Task 1: roles + bucket + repo + seed files.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
codecommit = boto3.client(
    "codecommit",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
codebuild = boto3.client(
    "codebuild",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
codepipeline = boto3.client(
    "codepipeline",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
logs = boto3.client(
    "logs",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
cw = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sns = boto3.client(
    "sns",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Verify CodeCommit is enabled on this account.
try:
    codecommit.list_repositories(maxResults=1)
except ClientError as e:
    if e.response["Error"]["Code"] in ("AccessDeniedException", "UnrecognizedClientException"):
        print(
            "CodeCommit is not available on this account (closed to new customers). "
            "Use the GitHub-via-CodeStar-Connections pattern instead and substitute the "
            "source action in Task 3."
        )
        raise SystemExit(1)
    raise

# Bucket.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Created bucket: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists, reusing.")
    else:
        raise
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

# Seed raw data.
seed_csv = "label,f1,f2,f3,f4\n" + "\n".join(
    f"{i % 2},{(i % 7) / 10:.3f},{(i % 11) / 10:.3f},{(i % 13) / 10:.3f},{(i % 17) / 10:.3f}"
    for i in range(200)
)
s3.put_object(Bucket=BUCKET_NAME, Key="raw/seed.csv", Body=seed_csv.encode("utf-8"))

# Trust + permissions for the three roles.
trust_cp = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "codepipeline.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
trust_cb = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "codebuild.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
trust_sm = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
broad_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:*", "codecommit:*", "codebuild:*", "codepipeline:*",
                "sagemaker:*", "iam:PassRole", "logs:*", "cloudwatch:*",
                "events:*", "sns:*",
            ],
            "Resource": "*",
        }
    ],
}
for role_name, trust, policy_name in (
    (PIPELINE_ROLE_NAME, trust_cp, PIPELINE_INLINE_POLICY),
    (BUILD_ROLE_NAME, trust_cb, BUILD_INLINE_POLICY),
    (SM_ROLE_NAME, trust_sm, SM_INLINE_POLICY),
):
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust),
            Description=f"labrun mla lab 12 role: {role_name}",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise
    iam.put_role_policy(
        RoleName=role_name, PolicyName=policy_name, PolicyDocument=json.dumps(broad_policy)
    )
print("Your IAM roles are stuck in traffic, give them 10 seconds...")
time.sleep(10)

# Repository.
try:
    repo = codecommit.create_repository(
        repositoryName=REPO_NAME,
        repositoryDescription="labrun mla lab 12 retraining repo",
        tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    REPO_CLONE_URL_HTTP = repo["repositoryMetadata"]["cloneUrlHttp"]
    print(f"Created repository: {REPO_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "RepositoryNameExistsException":
        existing = codecommit.get_repository(repositoryName=REPO_NAME)
        REPO_CLONE_URL_HTTP = existing["repositoryMetadata"]["cloneUrlHttp"]
        print(f"Repository {REPO_NAME} already exists, reusing.")
    else:
        raise

buildspec_yaml = f'''version: 0.2
phases:
  install:
    runtime-versions:
      python: 3.11
    commands:
      - pip install --quiet sagemaker==2.232.0 boto3
  build:
    commands:
      - echo Upserting and starting the SageMaker Pipeline...
      - python pipeline_definition.py upsert-and-start
      - echo Done.
'''

pipeline_definition_py = f'''import json
import os
import sys
import boto3

REGION = "{REGION}"
BUCKET = "{BUCKET_NAME}"
PIPELINE_NAME = "{SM_PIPELINE_NAME}"
PACKAGE_GROUP = "{PACKAGE_GROUP_NAME}"
ROLE_ARN = "{SM_ROLE_ARN}"

def build_pipeline():
    import sagemaker
    from sagemaker.workflow.pipeline import Pipeline
    from sagemaker.workflow.pipeline_context import PipelineSession
    from sagemaker.workflow.steps import ProcessingStep, TrainingStep
    from sagemaker.workflow.step_collections import RegisterModel
    from sagemaker.estimator import Estimator
    from sagemaker.inputs import TrainingInput
    from sagemaker.sklearn.processing import SKLearnProcessor
    from sagemaker.processing import ProcessingInput, ProcessingOutput

    boto_sess = boto3.Session(region_name=REGION)
    pipeline_session = PipelineSession(boto_session=boto_sess)
    xgb_image = sagemaker.image_uris.retrieve(framework="xgboost", region=REGION, version="1.5-1")

    proc = SKLearnProcessor(
        framework_version="1.0-1",
        instance_type="ml.m5.large",
        instance_count=1,
        role=ROLE_ARN,
        sagemaker_session=pipeline_session,
    )
    proc_args = proc.run(
        code=f"s3://{{BUCKET}}/code/passthrough.py",
        inputs=[ProcessingInput(source=f"s3://{{BUCKET}}/raw/seed.csv", destination="/opt/ml/processing/input/raw")],
        outputs=[ProcessingOutput(output_name="train", source="/opt/ml/processing/train")],
    )
    split_step = ProcessingStep(name="ProcessStep", step_args=proc_args)

    est = Estimator(
        image_uri=xgb_image,
        instance_type="ml.m5.large",
        instance_count=1,
        role=ROLE_ARN,
        output_path=f"s3://{{BUCKET}}/output/train/",
        sagemaker_session=pipeline_session,
        hyperparameters={{"objective": "binary:logistic", "num_round": "20", "max_depth": "3", "eta": "0.2"}},
    )
    train_args = est.fit(inputs={{"train": TrainingInput(
        s3_data=split_step.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
        content_type="text/csv",
    )}})
    train_step = TrainingStep(name="TrainStep", step_args=train_args)

    register_step = RegisterModel(
        name="RegisterModelStep",
        estimator=est,
        model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.large"],
        transform_instances=["ml.m5.large"],
        model_package_group_name=PACKAGE_GROUP,
        approval_status="PendingManualApproval",
    )

    pipeline = Pipeline(
        name=PIPELINE_NAME,
        steps=[split_step, train_step, register_step],
        sagemaker_session=pipeline_session,
    )
    return pipeline

if __name__ == "__main__":
    arg = sys.argv[1] if len(sys.argv) > 1 else ""
    sm_c = boto3.client("sagemaker", region_name=REGION)
    try:
        sm_c.create_model_package_group(ModelPackageGroupName=PACKAGE_GROUP)
    except sm_c.exceptions.ClientError:
        pass
    pipeline = build_pipeline()
    if arg == "upsert-and-start":
        pipeline.upsert(role_arn=ROLE_ARN)
        exec_resp = pipeline.start()
        print(f"Started pipeline execution: {{exec_resp.arn}}")
    else:
        print(json.dumps(json.loads(pipeline.definition()), indent=2))
'''

# Upload a passthrough processing script the SM pipeline will use.
passthrough_py = '''import os
import shutil
raw_dir = "/opt/ml/processing/input/raw"
out_dir = "/opt/ml/processing/train"
os.makedirs(out_dir, exist_ok=True)
for f in os.listdir(raw_dir):
    if f.endswith(".csv"):
        with open(os.path.join(raw_dir, f)) as src, open(os.path.join(out_dir, "train.csv"), "w") as dst:
            next(src)  # drop header
            for line in src:
                dst.write(line)
        break
'''
s3.put_object(Bucket=BUCKET_NAME, Key="code/passthrough.py", Body=passthrough_py.encode("utf-8"))
print("Uploaded code/passthrough.py to S3.")

# Seed buildspec.yml and pipeline_definition.py to main.
# YOUR CODE: call codecommit.put_file(repositoryName=REPO_NAME,
# branchName="main", filePath="buildspec.yml",
# fileContent=buildspec_yaml.encode("utf-8"),
# commitMessage="add buildspec") -- but be careful: the first commit
# to an empty repo cannot specify a parent commit, while subsequent
# commits MUST specify the latest parentCommitId. Use the try/except
# pattern: catch ParentCommitIdRequiredException and pass
# parentCommitId from get_branch.
def put_file_safe(path: str, content: bytes, msg: str):
    try:
        codecommit.put_file(
            repositoryName=REPO_NAME,
            branchName="main",
            fileContent=content,
            filePath=path,
            commitMessage=msg,
        )
    except ClientError as e:
        if e.response["Error"]["Code"] in (
            "ParentCommitIdRequiredException", "InvalidParentCommitIdException"
        ):
            parent = codecommit.get_branch(repositoryName=REPO_NAME, branchName="main")[
                "branch"
            ]["commitId"]
            codecommit.put_file(
                repositoryName=REPO_NAME,
                branchName="main",
                fileContent=content,
                filePath=path,
                parentCommitId=parent,
                commitMessage=msg,
            )
        elif e.response["Error"]["Code"] == "SameFileContentException":
            pass
        else:
            raise

# YOUR CODE: call put_file_safe("buildspec.yml", buildspec_yaml.encode("utf-8"),
#                                "add buildspec")
# and  put_file_safe("pipeline_definition.py",
#                     pipeline_definition_py.encode("utf-8"),
#                     "add pipeline definition")

print("Seeded buildspec.yml and pipeline_definition.py to main.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: repo exists with both seed files on main and the lab tag.

def checkpoint_1(session):
    try:
        cc = boto3.client(
            "codecommit",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            cc.get_repository(repositoryName=REPO_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Repository {REPO_NAME} not found: {e}",
            )
        for path in ("buildspec.yml", "pipeline_definition.py"):
            try:
                resp = cc.get_file(
                    repositoryName=REPO_NAME, filePath=path, commitSpecifier="main",
                )
                if not resp.get("fileContent"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"File {path} on main is empty.",
                    )
                content = resp["fileContent"].decode("utf-8")
                if path == "buildspec.yml" and "pipeline_definition.py" not in content:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            "buildspec.yml does not reference pipeline_definition.py. "
                            "The build phase must call python pipeline_definition.py."
                        ),
                    )
                if path == "pipeline_definition.py" and "pipeline.upsert" not in content:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            "pipeline_definition.py does not call pipeline.upsert(...). "
                            "The CodeBuild step must upsert the SageMaker Pipeline."
                        ),
                    )
            except ClientError:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"File {path} not found on main branch.",
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The `put_file_safe` helper handles the empty-repo-vs-existing-repo parent-commit-id branching. You only need to call it twice, once per file.

</details>

<details><summary>Hint 2 (direction)</summary>

`put_file_safe(path, content_bytes, commit_msg)`. The `buildspec_yaml` and `pipeline_definition_py` strings are pre-built; encode them to UTF-8 before passing.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the two put_file_safe calls):

```python
put_file_safe("buildspec.yml", buildspec_yaml.encode("utf-8"), "add buildspec")
put_file_safe(
    "pipeline_definition.py",
    pipeline_definition_py.encode("utf-8"),
    "add pipeline definition",
)
```

</details>

**Wallet check.** CodeCommit is free for the first 5 active users. The repo plus 2 files plus 200 KB of total storage is well under the $1/active-user/month line and 6 GB-month free-tier ceiling. IAM roles are free. S3 puts are fractions of a penny. Damage so far: about $0.00. Your coffee dominates.

## Task 2: Create the CodeBuild project pointing at the repo with the build role attached

The CodeBuild project reads from `Source.Type=CODECOMMIT` with the repo URL and the `buildspec.yml` path. It runs on a small Linux container (`aws/codebuild/standard:7.0`) on `BUILD_GENERAL1_SMALL`. The service role is the build-role you created in Task 1.

In [ ]:
# NBVAL_SKIP
# Task 2: CodeBuild project.

source_config = {
    "type": "CODECOMMIT",
    "location": REPO_CLONE_URL_HTTP,
    "buildspec": "buildspec.yml",
}
artifacts_config = {"type": "NO_ARTIFACTS"}
env_config = {
    "type": "LINUX_CONTAINER",
    "image": "aws/codebuild/standard:7.0",
    "computeType": "BUILD_GENERAL1_SMALL",
}

# YOUR CODE: call codebuild.create_project(name=BUILD_PROJECT,
# source=source_config, artifacts=artifacts_config, environment=env_config,
# serviceRole=BUILD_ROLE_ARN, tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}]).
# Catch ResourceAlreadyExistsException and ignore.
try:
    pass
except ClientError as e:
    if e.response["Error"]["Code"] != "ResourceAlreadyExistsException":
        raise
    print(f"CodeBuild project {BUILD_PROJECT} already exists, reusing.")
print(f"CodeBuild project {BUILD_PROJECT} created.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: CodeBuild project exists with correct source + role.

def checkpoint_2(session):
    try:
        cb = boto3.client(
            "codebuild",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        resp = cb.batch_get_projects(names=[BUILD_PROJECT])
        projects = resp.get("projects", [])
        if not projects:
            return CheckpointResult(
                status="fail",
                error_reason=f"CodeBuild project {BUILD_PROJECT} was not found.",
            )
        proj = projects[0]
        if proj.get("source", {}).get("type") != "CODECOMMIT":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CodeBuild source.type is {proj.get('source', {}).get('type')!r}, "
                    f"expected 'CODECOMMIT'."
                ),
            )
        if proj.get("serviceRole") != BUILD_ROLE_ARN:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CodeBuild serviceRole is {proj.get('serviceRole')!r}, "
                    f"expected {BUILD_ROLE_ARN!r}."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

`create_project` takes the source, artifacts, environment, service role, and tags. All five dicts and the role ARN are already in scope.

</details>

<details><summary>Hint 2 (direction)</summary>

`codebuild.create_project(name=BUILD_PROJECT, source=source_config, artifacts=artifacts_config, environment=env_config, serviceRole=BUILD_ROLE_ARN, tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}])` is the full call. CodeBuild tags use lowercase `key`/`value` keys (not capitalized like most other AWS APIs).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the create_project call inside try/except):

```python
try:
    codebuild.create_project(
        name=BUILD_PROJECT,
        source=source_config,
        artifacts=artifacts_config,
        environment=env_config,
        serviceRole=BUILD_ROLE_ARN,
        tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "ResourceAlreadyExistsException":
        raise
    print(f"CodeBuild project {BUILD_PROJECT} already exists, reusing.")
```

</details>

**Wallet check.** CodeBuild project creation is free. CodeBuild bills only while a build runs. Damage so far: about $0.00. Coffee, still ahead by a wide margin.

## Task 3: Create the CodePipeline with Source + Build + Approve stages and trigger the first execution

The pipeline has three stages: a Source stage pulling from CodeCommit main, a Build stage running the CodeBuild project, and an Approve stage with a `Manual_Approval` action. The Approve action does not actually deploy; it is the audit pattern.

After `create_pipeline` returns, CodePipeline automatically triggers the first execution. You can also call `start_pipeline_execution` explicitly to skip the wait. Capture the executionId so Task 5 can approve it.

In [ ]:
# NBVAL_SKIP
# Task 3: CodePipeline.

pipeline_def = {
    "name": PIPELINE_NAME,
    "roleArn": PIPELINE_ROLE_ARN,
    "artifactStore": {"type": "S3", "location": BUCKET_NAME},
    "stages": [
        {
            "name": "Source",
            "actions": [{
                "name": "CodeCommitSource",
                "actionTypeId": {
                    "category": "Source",
                    "owner": "AWS",
                    "provider": "CodeCommit",
                    "version": "1",
                },
                "configuration": {
                    "RepositoryName": REPO_NAME,
                    "BranchName": "main",
                    "PollForSourceChanges": "true",
                },
                "outputArtifacts": [{"name": "SourceArtifact"}],
                "runOrder": 1,
            }],
        },
        {
            "name": "Build",
            "actions": [{
                "name": "CodeBuildAction",
                "actionTypeId": {
                    "category": "Build",
                    "owner": "AWS",
                    "provider": "CodeBuild",
                    "version": "1",
                },
                "configuration": {"ProjectName": BUILD_PROJECT},
                "inputArtifacts": [{"name": "SourceArtifact"}],
                "runOrder": 1,
            }],
        },
        {
            "name": "Approve",
            "actions": [{
                "name": "ManualApproval",
                "actionTypeId": {
                    "category": "Approval",
                    "owner": "AWS",
                    "provider": "Manual",
                    "version": "1",
                },
                "configuration": {},
                "runOrder": 1,
            }],
        },
    ],
}

# YOUR CODE: call codepipeline.create_pipeline(pipeline=pipeline_def,
# tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}]). Catch
# PipelineNameInUseException and skip.
try:
    pass
except ClientError as e:
    if e.response["Error"]["Code"] != "PipelineNameInUseException":
        raise
    print(f"Pipeline {PIPELINE_NAME} already exists, reusing.")
print(f"Pipeline {PIPELINE_NAME} created.")

# Trigger an explicit execution.
time.sleep(5)
exec_resp = codepipeline.start_pipeline_execution(name=PIPELINE_NAME)
PIPELINE_EXECUTION_ID = exec_resp["pipelineExecutionId"]
print(f"Started pipeline execution: {PIPELINE_EXECUTION_ID}")
print("Pipeline is teaching CodeBuild how SageMaker thinks, give it 12 to 15 minutes...")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: pipeline exists with the three expected stages.

def checkpoint_3(session):
    try:
        cp = boto3.client(
            "codepipeline",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            d = cp.get_pipeline(name=PIPELINE_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Pipeline {PIPELINE_NAME} not found: {e}",
            )
        stages = d["pipeline"]["stages"]
        names = [s["name"] for s in stages]
        expected = ["Source", "Build", "Approve"]
        if names != expected:
            return CheckpointResult(
                status="fail",
                error_reason=f"Pipeline stages are {names!r}, expected {expected!r}.",
            )
        source_action = stages[0]["actions"][0]["actionTypeId"]
        if source_action["provider"] != "CodeCommit":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Source stage provider is {source_action['provider']!r}, expected "
                    f"'CodeCommit'."
                ),
            )
        build_action = stages[1]["actions"][0]["actionTypeId"]
        if build_action["provider"] != "CodeBuild":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Build stage provider is {build_action['provider']!r}, expected "
                    f"'CodeBuild'."
                ),
            )
        approve_action = stages[2]["actions"][0]["actionTypeId"]
        if approve_action["category"] != "Approval":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Approve stage action category is {approve_action['category']!r}, "
                    f"expected 'Approval'."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The `pipeline_def` dict is built; pass it under `pipeline=...` to `create_pipeline`.

</details>

<details><summary>Hint 2 (direction)</summary>

`codepipeline.create_pipeline(pipeline=pipeline_def, tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}])`. CodePipeline tags use the lowercase `key`/`value` keys.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the create_pipeline call inside try/except):

```python
try:
    codepipeline.create_pipeline(
        pipeline=pipeline_def,
        tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response["Error"]["Code"] != "PipelineNameInUseException":
        raise
    print(f"Pipeline {PIPELINE_NAME} already exists, reusing.")
```

</details>

**Wallet check.** CodePipeline at $1 per active pipeline per month starts billing the moment the pipeline exists. Prorated to a 90-minute session that is $0.002. The execution that just started will run CodeBuild for about 3 minutes ($0.015) plus the SageMaker Pipeline 4-step run on ml.m5.large for about 12 minutes total ($0.023). Total damage so far: about $0.04 with the execution still in flight. Coffee, still safely ahead.

## Task 4: Create the EventBridge rule on `SageMaker Model Package State Change` and route to a CloudWatch log group

When the SageMaker Pipeline finishes registering the model, SageMaker emits a `Model Package State Change` event to the default EventBridge bus. The rule below matches on `source=aws.sagemaker` and `detail-type=SageMaker Model Package State Change`. The target is a CloudWatch log group; in production the target would be Lambda or SNS for actual notification.

Create the log group first (EventBridge needs the target to exist before `put_targets`). Then create the rule with the event pattern. Then call `put_targets` to attach the log group ARN.

In [ ]:
# NBVAL_SKIP
# Task 4: EventBridge rule + CloudWatch log group target.

try:
    logs.create_log_group(
        logGroupName=LOG_GROUP_NAME, tags={LAB_TAG_KEY: LAB_TAG_VALUE}
    )
    print(f"Created log group: {LOG_GROUP_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        print(f"Log group {LOG_GROUP_NAME} already exists, reusing.")
    else:
        raise
logs.put_retention_policy(logGroupName=LOG_GROUP_NAME, retentionInDays=7)

event_pattern = {
    "source": ["aws.sagemaker"],
    "detail-type": ["SageMaker Model Package State Change"],
}

# YOUR CODE: call events.put_rule(Name=EB_RULE_NAME,
# EventPattern=json.dumps(event_pattern), State="ENABLED",
# Description="labrun mla lab 12 model package state change",
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]).

log_group_arn = (
    f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{LOG_GROUP_NAME}:*"
)
events.put_targets(
    Rule=EB_RULE_NAME,
    Targets=[{"Id": "1", "Arn": log_group_arn.rstrip(":*")}],
)
print(f"EventBridge rule {EB_RULE_NAME} created and target attached.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: rule exists with the right event pattern + target.

def checkpoint_4(session):
    try:
        ev = boto3.client(
            "events",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            d = ev.describe_rule(Name=EB_RULE_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"EventBridge rule {EB_RULE_NAME} not found: {e}",
            )
        pattern_str = d.get("EventPattern") or "{}"
        pat = json.loads(pattern_str)
        if "aws.sagemaker" not in pat.get("source", []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventPattern source is {pat.get('source')!r}, expected "
                    f"to include 'aws.sagemaker'."
                ),
            )
        if "SageMaker Model Package State Change" not in pat.get("detail-type", []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventPattern detail-type is {pat.get('detail-type')!r}, expected "
                    f"to include 'SageMaker Model Package State Change'."
                ),
            )
        targets = ev.list_targets_by_rule(Rule=EB_RULE_NAME).get("Targets", [])
        if not targets:
            return CheckpointResult(
                status="fail",
                error_reason="EventBridge rule has no targets attached.",
            )
        if "log-group" not in targets[0]["Arn"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EventBridge rule target ARN is {targets[0]['Arn']!r}, expected a "
                    f"CloudWatch log group ARN."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

`put_rule` takes the rule name, the event pattern as a JSON string, and the state (ENABLED). The lab already built the event pattern dict.

</details>

<details><summary>Hint 2 (direction)</summary>

Pass `EventPattern=json.dumps(event_pattern)` (the API wants a string, not a dict). `State="ENABLED"` is the default but explicit is clearer. Tags use the capitalized `Key`/`Value` shape on the EventBridge API.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the put_rule call):

```python
events.put_rule(
    Name=EB_RULE_NAME,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="labrun mla lab 12 model package state change",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** EventBridge bills $1 per million events. This lab's SageMaker pipeline emits roughly 5 events. Free. CloudWatch log groups are billed on ingestion plus storage; 5 events in 7-day retention is fractions of a penny. Damage so far: about $0.04 (still mostly the in-flight CodePipeline execution).

## Task 5: Create the CloudWatch alarm, wait for the pipeline to reach the Approve stage, then approve it

The CloudWatch alarm watches the `FailedPipelineExecutions` metric on the AWS/CodePipeline namespace. Threshold is 1 over a 5-minute period. When the alarm trips it publishes to an SNS topic (no subscribers wired in this lab; in production the topic would fan out to email or PagerDuty).

Then wait for the CodePipeline execution to reach the Approve stage. When it lands there, call `put_approval_result` to simulate the manual approval. The pipeline reaches `Succeeded` once approved.

Your job: write the put_approval_result call once the approve action token is captured.

In [ ]:
# NBVAL_SKIP
# Task 5: SNS topic + CloudWatch alarm + wait for Approve + approve.

try:
    sns_resp = sns.create_topic(
        Name=SNS_TOPIC_NAME, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]
    )
    SNS_TOPIC_ARN = sns_resp["TopicArn"]
    print(f"Created SNS topic: {SNS_TOPIC_ARN}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("AlreadyExistsException", "InvalidParameter"):
        print(f"SNS topic {SNS_TOPIC_NAME} already exists.")
    else:
        raise

cw.put_metric_alarm(
    AlarmName=ALARM_NAME,
    AlarmDescription="labrun mla lab 12 CodePipeline failure alarm",
    Namespace="AWS/CodePipeline",
    MetricName="FailedPipelineExecutions",
    Statistic="Sum",
    Dimensions=[{"Name": "PipelineName", "Value": PIPELINE_NAME}],
    Period=300,
    EvaluationPeriods=1,
    Threshold=1,
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[SNS_TOPIC_ARN],
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"Created CloudWatch alarm: {ALARM_NAME}")

print("Waiting for the pipeline execution to reach the Approve stage...")
approval_token = None
elapsed = 0
approve_status = None
while elapsed < 1500:
    state = codepipeline.get_pipeline_state(name=PIPELINE_NAME)
    for stage in state["stageStates"]:
        if stage["stageName"] == "Approve":
            for action in stage.get("actionStates", []):
                if action["actionName"] == "ManualApproval":
                    latest = action.get("latestExecution", {})
                    if latest.get("status") == "InProgress":
                        approval_token = latest.get("token")
                        approve_status = "InProgress"
                    elif latest.get("status") == "Succeeded":
                        approve_status = "Succeeded"
    if approval_token or approve_status == "Succeeded":
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(f"  {elapsed}s elapsed, waiting for Approve stage...")

if approve_status == "Succeeded":
    print("Approve stage already succeeded (likely a re-run).")
elif approval_token:
    # YOUR CODE: call codepipeline.put_approval_result(
    #     pipelineName=PIPELINE_NAME,
    #     stageName="Approve",
    #     actionName="ManualApproval",
    #     result={"summary": "Auto-approved by labrun lab", "status": "Approved"},
    #     token=approval_token,
    # ) to release the gate.
    print("Approval submitted.")
else:
    print("Approval token not found within the 25-minute ceiling.")
    print("Inspect the CodePipeline state in the AWS console.")

# Wait for Succeeded.
print("Waiting for the pipeline to reach Succeeded...")
elapsed = 0
exec_status = None
while elapsed < 600:
    d = codepipeline.get_pipeline_execution(
        pipelineName=PIPELINE_NAME, pipelineExecutionId=PIPELINE_EXECUTION_ID
    )
    exec_status = d["pipelineExecution"]["status"]
    if exec_status in ("Succeeded", "Failed", "Stopped", "Cancelled"):
        break
    time.sleep(20)
    elapsed += 20
print(f"Pipeline execution final status: {exec_status}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: alarm exists, pipeline reached Succeeded, alarm state OK.

def checkpoint_5(session):
    try:
        cw_c = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        cp = boto3.client(
            "codepipeline",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        alarms = cw_c.describe_alarms(AlarmNames=[ALARM_NAME]).get("MetricAlarms", [])
        if not alarms:
            return CheckpointResult(
                status="fail",
                error_reason=f"Alarm {ALARM_NAME} not found.",
            )
        alarm_state = alarms[0].get("StateValue")
        if alarm_state == "ALARM":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Alarm state is {alarm_state!r}. The pipeline failure metric "
                    f"crossed threshold; investigate failed executions."
                ),
            )
        d = cp.get_pipeline_execution(
            pipelineName=PIPELINE_NAME, pipelineExecutionId=PIPELINE_EXECUTION_ID
        )
        status = d["pipelineExecution"]["status"]
        if status != "Succeeded":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Pipeline execution status is {status!r}, expected 'Succeeded'. "
                    f"Confirm put_approval_result was called with the right token."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

`put_approval_result` takes the pipeline name, the stage name ("Approve"), the action name ("ManualApproval"), the result dict, and the token captured from `get_pipeline_state`.

</details>

<details><summary>Hint 2 (direction)</summary>

The result dict has `summary` (free-text reason) and `status` ("Approved" or "Rejected"). The token comes from the `latestExecution.token` field on the action state.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5 (the put_approval_result call):

```python
codepipeline.put_approval_result(
    pipelineName=PIPELINE_NAME,
    stageName="Approve",
    actionName="ManualApproval",
    result={
        "summary": "Auto-approved by labrun lab",
        "status": "Approved",
    },
    token=approval_token,
)
```

</details>

**Wallet check.** CloudWatch alarms are free below 10 per month. SNS topics with no subscribers are free at this volume. CodePipeline at the prorated rate continues to tick. Total damage so far: about $0.06 to $0.08 with the in-flight pipeline run plus the CodeBuild minutes plus the SageMaker Pipeline steps.

In [ ]:
# NBVAL_SKIP
# Cleanup. The longest cleanup in the MLA track. Order:
# 1. Stop any in-flight CodePipeline execution.
# 2. Delete CloudWatch alarm.
# 3. Delete SNS topic.
# 4. Remove EventBridge targets, delete rule.
# 5. Delete CloudWatch log group.
# 6. Delete CodePipeline ($1/month residual stops here).
# 7. Delete CodeBuild project.
# 8. Delete SageMaker pipeline + model packages + group.
# 9. Delete CodeCommit repository.
# 10. Hand off to run_cleanup for IAM roles + S3 bucket.

manual_warnings = []


def _safe(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in (
            "ResourceNotFoundException",
            "ResourceNotFound",
            "NotFoundException",
            "RepositoryDoesNotExistException",
            "PipelineNotFoundException",
            "ValidationException",
            "NoSuchEntity",
            "NotFound",
        ):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


# Stop in-flight CodePipeline execution if any.
try:
    state = codepipeline.get_pipeline_state(name=PIPELINE_NAME)
    for stage in state["stageStates"]:
        latest = stage.get("latestExecution", {})
        if latest.get("status") == "InProgress":
            exec_id = latest.get("pipelineExecutionId")
            if exec_id:
                try:
                    codepipeline.stop_pipeline_execution(
                        pipelineName=PIPELINE_NAME,
                        pipelineExecutionId=exec_id,
                        abandon=True,
                        reason="labrun cleanup",
                    )
                    print(f"Requested stop on execution {exec_id}.")
                except ClientError:
                    pass
except ClientError:
    pass

_safe(
    f"alarm {ALARM_NAME}",
    lambda: cw.delete_alarms(AlarmNames=[ALARM_NAME]),
    f"aws cloudwatch delete-alarms --alarm-names {ALARM_NAME}",
)
_safe(
    f"SNS topic {SNS_TOPIC_ARN}",
    lambda: sns.delete_topic(TopicArn=SNS_TOPIC_ARN),
    f"aws sns delete-topic --topic-arn {SNS_TOPIC_ARN}",
)
_safe(
    f"event rule targets {EB_RULE_NAME}",
    lambda: events.remove_targets(Rule=EB_RULE_NAME, Ids=["1"]),
    f"aws events remove-targets --rule {EB_RULE_NAME} --ids 1",
)
_safe(
    f"event rule {EB_RULE_NAME}",
    lambda: events.delete_rule(Name=EB_RULE_NAME),
    f"aws events delete-rule --name {EB_RULE_NAME}",
)
_safe(
    f"log group {LOG_GROUP_NAME}",
    lambda: logs.delete_log_group(logGroupName=LOG_GROUP_NAME),
    f"aws logs delete-log-group --log-group-name {LOG_GROUP_NAME}",
)

pipeline_gone = False
_safe(
    f"CodePipeline {PIPELINE_NAME}",
    lambda: codepipeline.delete_pipeline(name=PIPELINE_NAME),
    f"aws codepipeline delete-pipeline --name {PIPELINE_NAME}",
)
pipeline_gone = True  # delete_pipeline returns synchronously; idempotent

_safe(
    f"CodeBuild project {BUILD_PROJECT}",
    lambda: codebuild.delete_project(name=BUILD_PROJECT),
    f"aws codebuild delete-project --name {BUILD_PROJECT}",
)

# SageMaker pipeline + packages.
_safe(
    f"sagemaker pipeline {SM_PIPELINE_NAME}",
    lambda: sm.delete_pipeline(PipelineName=SM_PIPELINE_NAME),
    f"aws sagemaker delete-pipeline --pipeline-name {SM_PIPELINE_NAME}",
)
try:
    pkgs = sm.list_model_packages(
        ModelPackageGroupName=PACKAGE_GROUP_NAME, MaxResults=50
    ).get("ModelPackageSummaryList", [])
except ClientError:
    pkgs = []
for p in pkgs:
    arn = p["ModelPackageArn"]
    _safe(
        f"model package {arn}",
        lambda a=arn: sm.delete_model_package(ModelPackageName=a),
        f"aws sagemaker delete-model-package --model-package-name {arn}",
    )
_safe(
    f"model package group {PACKAGE_GROUP_NAME}",
    lambda: sm.delete_model_package_group(ModelPackageGroupName=PACKAGE_GROUP_NAME),
    f"aws sagemaker delete-model-package-group --model-package-group-name {PACKAGE_GROUP_NAME}",
)

_safe(
    f"CodeCommit repo {REPO_NAME}",
    lambda: codecommit.delete_repository(repositoryName=REPO_NAME),
    f"aws codecommit delete-repository --repository-name {REPO_NAME}",
)

# Hand off to run_cleanup.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 1  # CodePipeline ($1/month residual)
critical_destroyed = 1 if pipeline_gone else 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.05 to $0.10.** CodePipeline at $1 per pipeline per month prorated to 90 minutes is $0.002. CodeBuild at $0.005 per build-minute on `general1.small` for 3 minutes is $0.015. The SageMaker Pipeline 4-step run on ml.m5.large at $0.115/hour for 3 minutes per step is $0.023. EventBridge, CloudWatch, and SNS at this volume are free. S3 storage and PUT operations are fractions of a penny. If you confirmed in the cleanup summary that the CodePipeline is gone, the $1/month residual stopped. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the AWS-recommended MLOps CI/CD pattern this lab implements. CodeCommit holds the pipeline definition. CodeBuild upserts the SageMaker Pipeline and triggers a run. SageMaker Pipelines runs the four-step train + evaluate + register chain. EventBridge listens for the Model Registry state change. CloudWatch alarms on pipeline failure. Manual approval gates the deploy. Each piece could be replaced (GitHub instead of CodeCommit, Jenkins instead of CodeBuild, Step Functions instead of SageMaker Pipelines, SNS instead of EventBridge). What changes if you swap each piece, and what do you give up?

2. The Approve stage in this lab does not actually deploy to a staging endpoint; the approval is a no-op for cost reasons. Walk through what a real production approve-then-deploy stage looks like: which AWS service deploys (CodeDeploy, SageMaker direct, a custom Lambda), what blue/green vs canary vs linear strategies AWS supports natively for SageMaker endpoints, and how the EventBridge event from this lab feeds a rollback in case of a bad deploy.

## Exam-Style Practice

**Question 1 (Domain 3, MLOps CI/CD pattern selection):**

A platform team is wiring CI/CD for an ML retraining workflow. Requirements: source control with pull-request review, automated build on commit, automated SageMaker Pipelines invocation, model-registry registration, EventBridge notification on registration, and an automated CloudWatch alarm on pipeline failure. The team wants AWS-native services with the least cross-service glue code. Which combination fits?

A. GitHub for source + Jenkins for build + Step Functions for orchestration + SNS for notification + Datadog for alarms.

B. CodeCommit + CodeBuild + SageMaker Pipelines + EventBridge + CloudWatch alarms (the AWS-native MLOps pattern this lab implements).

C. CodeCommit + CodeDeploy + AWS Batch + SQS + CloudWatch alarms.

D. CodeCommit + CodeBuild + AWS Glue + Lambda + CloudWatch alarms.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong on 'AWS-native, least glue': GitHub and Jenkins are non-AWS services; integrating them with SageMaker requires custom adapters. Step Functions can orchestrate SageMaker but lacks the ML-specific step types Pipelines provides. The combination works but adds glue.
- B is correct: the AWS-native MLOps pattern. CodeCommit for source (with Git semantics for PR review), CodeBuild for build (the AWS-recommended CI runner), SageMaker Pipelines for the ML orchestration with native ProcessingStep / TrainingStep / RegisterModelStep, EventBridge for the SageMaker Model Package State Change event, CloudWatch alarms on the CodePipeline failure metric. All services are AWS-managed with native integration.
- C is wrong: CodeDeploy is for application deployment (EC2, ECS, Lambda), not for ML pipeline orchestration. AWS Batch is for batch compute jobs, not the right tool for SageMaker pipeline runs.
- D is wrong: AWS Glue is an ETL service, not an ML pipeline orchestrator. Lambda can stitch services together but adds glue code where Pipelines provides it natively.

</details>

**Question 2 (Domain 4, EventBridge-driven retraining):**

A team wants to automatically trigger SageMaker Pipelines retraining whenever a Model Monitor data-quality run reports `CompletedWithViolations`. Which AWS-native pattern fits with the least glue code?

A. EventBridge rule on `Model Monitor execution state changes` filtered to `CompletedWithViolations`, with a target that invokes a Lambda which calls `sagemaker.start_pipeline_execution`.

B. CloudWatch Events polling every 5 minutes via Lambda to check Model Monitor execution status and trigger retraining manually.

C. Step Functions state machine that polls Model Monitor every hour and triggers retraining when violations are detected.

D. SageMaker Pipelines built-in retrain trigger on Model Monitor violations.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: SageMaker emits `Model Monitor execution state change` events to EventBridge natively. An EventBridge rule with an event pattern filtered to the violation state and a Lambda target is the AWS-native, least-glue pattern. The Lambda calls `start_pipeline_execution` on the retraining pipeline.
- B is wrong on 'least glue': polling via Lambda is an anti-pattern when an event-driven option exists. EventBridge already publishes the state change as an event; polling reinvents what is already provided.
- C is wrong: Step Functions is an orchestrator, not the right tool for 'wait for a state change.' EventBridge handles the wait natively.
- D is wrong: SageMaker Pipelines does not have a built-in trigger on Model Monitor violations. The trigger comes from EventBridge, which Pipelines is invoked by (not the other way around).

</details>